# SPA613M Programming Assignment

You are part of an **IIT Kanpur** Celestial observing team studying **FRB 20220912A**, a repeating fast radio burst. This is one continuous assignment: it begins with units and time, moves through coordinate transformations and spherical calculations, develops the geometry and visibility of a source from GTC, and ends by answering progressively harder observing questions involving a FRB and a one-year all-sky GTC visibility infographic.

Use **UTC internally** for astronomical calculations. Whenever you report a civil clock time, show **IIT Kanpur time—Indian Standard Time (IST, `Asia/Kolkata`, UTC+05:30)** and retain UTC where requested.

**Total: 40 marks**  
**Optional:** Q20 carries 2 bonus marks.


## Marking Scheme

| Part | Questions | Topic | Marks |
|---|---:|---|---:|
| A | Q1–Q3 | Current time, quantities, conversions, JD/MJD, and IST | 5 |
| B | Q4–Q7 | Angles, coordinate systems, and angular separation | 8 |
| C | Q8–Q13 | GTC, LST, hour angle, transit, and altitude calculations | 12 |
| D | Q14–Q17 | Airmass, Sun, Moon, and practical visibility decisions | 9 |
| E | Q18 | Observing interval, GTC airmass figure, and interpretation | 4 |
| F | Q19 | Single-FRB visibility | 2 |
| Bonus extension | Q20 | One-year all-sky GTC animation | 2 bonus |
| **Assessed total** | **Q1–Q19** |  | **40** |


## Fixed target record

FRB 20220912A is a repeating fast radio burst. Use the following published VLBI position throughout the assignment:

- Source: `FRB 20220912A`
- ICRS/J2000 RA: `23h09m04.8989s`
- ICRS/J2000 Dec: `+48d42m23.9078s`
- Coordinate uncertainty: `5 mas`
- Dispersion measure: $219.46\,\mathrm{pc\,cm^{-3}}$
- Host redshift: $0.0771$

Source: Hewitt et al., *Milliarcsecond localization of the hyperactive repeating FRB 20220912A*, MNRAS 529, 1814–1830. The values above are frozen, so no catalogue or web query is required. [Published article](https://academic.oup.com/mnras/article/529/2/1814/7623035)

## Fixed GTC observing case

| Item | Course value |
|---|---|
| Observatory | Gran Telescopio Canarias (GTC) |
| Longitude | $-17.889^\circ$ |
| Latitude | $+28.758^\circ$ |
| Height | $2396\,\mathrm{m}$ |
| Calculation interval | `2026-08-30T12:00:00Z` through `2026-08-31T12:00:00Z` |
| Sampling | 10 minutes; both endpoints included; 145 samples |
| Minimum target altitude | $20^\circ$ |
| Permitted target airmass | finite $1\leq X\leq2.5$ |
| Astronomical darkness | Sun altitude $\leq-18^\circ$ |
| Minimum Moon separation | $30^\circ$ |

A target is observable only when **all four** target-altitude, target-airmass, Sun, and Moon-separation conditions pass simultaneously.

## Setup

Run the dependency cell before Q1. The setup supplies imports, fixed course constants, a reproducible offline astronomy configuration, an inclusive 145-sample UTC grid, IIT Kanpur formatting helpers, and a relative output directory.

In [1]:
import subprocess
import sys

REQUIRED_PACKAGES = [
    "numpy>=1.26",
    "matplotlib>=3.8",
    "astropy>=6.0",
    "astropy-iers-data",
    "tzdata",
    "healpy>=1.18",
    "astroplan>=0.10",
    "imageio>=2.31",
]

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        *REQUIRED_PACKAGES,
    ]
)
print("Dependencies installed: NumPy, Matplotlib, Astropy, IERS data, timezone data, healpy, astroplan, and imageio.")

Dependencies installed: NumPy, Matplotlib, Astropy, IERS data, timezone data, healpy, astroplan, and imageio.


In [2]:
from datetime import timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import astropy
import matplotlib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import healpy as hp
import numpy as np
from astropy import units as u
from astropy.coordinates import (
    AltAz,
    Angle,
    BarycentricTrueEcliptic,
    EarthLocation,
    SkyCoord,
    get_body,
    get_sun,
    solar_system_ephemeris,
)
from astropy.time import Time
from astropy.utils import iers
from astroplan import Observer

# Keep the astronomy calculation reproducible and independent of a live IERS download.
iers.conf.auto_download = False
iers.conf.auto_max_age = None

IITK_TIMEZONE = ZoneInfo("Asia/Kolkata")

FRB_NAME = "FRB 20220912A"
FRB_RA_J2000 = "23h09m04.8989s"
FRB_DEC_J2000 = "+48d42m23.9078s"
EARLIER_POSITION_RA = "23h09m04.9s"
EARLIER_POSITION_DEC = "+48d42m25.4s"

START_UTC = "2026-08-30T12:00:00"
STOP_UTC = "2026-08-31T12:00:00"
STEP_MINUTES = 10

GTC_LONGITUDE_DEG = -17.889
GTC_LATITUDE_DEG = 28.758
GTC_HEIGHT_M = 2396.0

MIN_TARGET_ALTITUDE_DEG = 20.0
MAX_AIRMASS = 2.5
MAX_SUN_ALTITUDE_DEG = -18.0
MIN_MOON_SEPARATION_DEG = 30.0

OUTPUT_DIR = Path("assignment_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The inclusive 24-hour course grid is provided so you can focus on astronomy.
start_time = Time(START_UTC, scale="utc")
stop_time = Time(STOP_UTC, scale="utc")
step = STEP_MINUTES * u.min
number_of_steps = int(
    np.round(((stop_time - start_time) / step).to_value(u.dimensionless_unscaled))
)
times = start_time + np.arange(number_of_steps + 1) * step
assert len(times) == 145

def format_iitk(time_value):
    """Return one Astropy Time as the IIT Kanpur civil clock."""
    return time_value.to_datetime(timezone=IITK_TIMEZONE).strftime(
        "%Y-%m-%d %H:%M:%S IST"
    )

def format_utc(time_value):
    """Return one Astropy Time with an explicit UTC label."""
    return time_value.utc.strftime("%Y-%m-%d %H:%M:%S UTC")

print("Target:", FRB_NAME)
print("Observatory: Gran Telescopio Canarias (GTC)")
print("Course grid:", len(times), "samples")
print("IIT Kanpur civil timezone:", IITK_TIMEZONE)
print("Astronomical calculations use UTC; civil-time answers also show IST.")

Target: FRB 20220912A
Observatory: Gran Telescopio Canarias (GTC)
Course grid: 145 samples
IIT Kanpur civil timezone: Asia/Kolkata
Astronomical calculations use UTC; civil-time answers also show IST.


# Part A.  Astropy foundations: time, units, and conversions (5 marks)

## Q1. Current UTC and IIT Kanpur time [1 mark]

Call `Time.now()` exactly once and store it as `now`. Print that same instant as ISO UTC and as the IIT Kanpur civil clock with an `IST` label. In one sentence, explain why two separate calls to `Time.now()` need not describe exactly the same instant.

In [3]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
now = Time.now()
print("current time (UTC):", format_utc(now))
print("current time (IIT Kanpur):", format_iitk(now))

current time (UTC): 2026-09-05 16:39:40 UTC
current time (IIT Kanpur): 2026-09-05 22:09:40 IST


Here, two Seprate calls for Time.now() give the diffrent values of times because each call at diffrent moment. Python takes time time to run each statement. so the code calls time.now() only once saved the result in now variable. by this way both utc and ist times are calculated from exactly the same moment.

## Q2. Physical quantities and unit conversions [2 marks]

Create Astropy quantities for the GTC height `2396 m`, wavelength `600 nm`, coordinate uncertainty `5 mas`, cadence `10 min`, and radio frequency `1.4 GHz`. Convert them respectively to kilometres, frequency in THz using `u.spectral()`, arcseconds, hours, and wavelength in centimetres using `u.spectral()`. Print every original value beside its converted value.

In [4]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.

gtc_height = 2396 * u.m
wavelength = 600 * u.nm
coord_uncertainty = 5 * u.mas
cadence = 10 * u.min
radio_frequency = 1.4 * u.GHz

# Conveted each value to corresponding giving units
gtc_height_km = gtc_height.to(u.km)
wavelength_thz = wavelength.to(u.THz, equivalencies=u.spectral())
coord_uncertainty_arcsec = coord_uncertainty.to(u.arcsec)
cadence_hr = cadence.to(u.hour)
radio_frequency_cm = radio_frequency.to(u.cm, equivalencies=u.spectral())

# Print original value along with converted value
print(f"GTC height:          {gtc_height} => {gtc_height_km}")
print(f"Wavelength:           {wavelength} => {wavelength_thz}")
print(f"Coordinate uncertainty: {coord_uncertainty} => {coord_uncertainty_arcsec}")
print(f"Cadence:              {cadence} => {cadence_hr}")
print(f"Radio frequency:      {radio_frequency} => {radio_frequency_cm}")

GTC height:          2396.0 m => 2.396 km
Wavelength:           600.0 nm => 499.6540966666666 THz
Coordinate uncertainty: 5.0 mas => 0.005 arcsec
Cadence:              10.0 min => 0.16666666666666666 h
Radio frequency:      1.4 GHz => 21.413747 cm


## Q3. UTC, JD, MJD, IST, and time arithmetic [2 marks]

Construct `planning_start = Time(START_UTC, scale="utc")`. Print UTC, Julian Date, Modified Julian Date, and IIT Kanpur time. Add `2.25 hour` with an Astropy quantity and print the later instant in UTC and IST. Finally verify that converting the original IST-aware Python datetime back into `Time` changes the instant by less than one microsecond.

In [8]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.

planning_start = Time(START_UTC, scale="utc")
print("Planning start (UTC):        ", format_utc(planning_start))
print("Planning start (Julian Date):", planning_start.jd)
print("Planning start (Mod. Julian Date):", planning_start.mjd)
print("Planning start (IIT Kanpur): ", format_iitk(planning_start))

# Add 2.25 hours using an Astropy Quantity
later_time = planning_start + 2.25 * u.hour
print()
print("Later time (UTC):        ", format_utc(later_time))
print("Later time (IIT Kanpur): ", format_iitk(later_time))

# check: IST-aware datetime -> Time, compare to original instant
ist_aware_datetime = planning_start.to_datetime(timezone=IITK_TIMEZONE)
roundtrip_time = Time(ist_aware_datetime)
difference = abs((roundtrip_time - planning_start).to(u.microsecond))
print()
print("Round-trip difference:", difference)
print("Verified: round-trip through IST-aware datetime preserves the instant to sub-microsecond precision.")

Planning start (UTC):         2026-08-30 12:00:00 UTC
Planning start (Julian Date): 2461283.0
Planning start (Mod. Julian Date): 61282.5
Planning start (IIT Kanpur):  2026-08-30 17:30:00 IST

Later time (UTC):         2026-08-30 14:15:00 UTC
Later time (IIT Kanpur):  2026-08-30 19:45:00 IST

Round-trip difference: 0.0 us
Verified: round-trip through IST-aware datetime preserves the instant to sub-microsecond precision.


# Part B. Angles and coordinate transformations (8 marks)


## Q4. Parse angles and construct an ICRS coordinate [2 marks]

Parse the target RA and Dec with `Angle`, then construct scalar `frb_icrs` with `SkyCoord`. Print RA and Dec in decimal degrees and padded sexagesimal notation. Verify RA $347.2704120833^\circ$ and Dec $+48.7066410556^\circ$ within $10^{-8}$ degree. Explain why RA commonly uses hours while Dec uses degrees.

In [10]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
frb_ra_angle = Angle(FRB_RA_J2000)
frb_dec_angle = Angle(FRB_DEC_J2000)
frb_icrs = SkyCoord(ra=frb_ra_angle, dec=frb_dec_angle, frame="icrs")

print("RA (decimal degrees):", frb_icrs.ra.deg, "deg")
print("Dec (decimal degrees):", frb_icrs.dec.deg, "deg")

print("RA  (padded sexagesimal):", frb_icrs.ra.to_string(unit=u.hourangle, sep=":", precision=4, pad=True))
print("Dec (padded sexagesimal):", frb_icrs.dec.to_string(unit=u.deg, sep=":", precision=4, pad=True, alwayssign=True))

# Verify against the expected decimal-degree values, within 10^-18 degree
expect_ra_deg = 347.2704120833
expect_dec_deg = 48.7066410556

ra_diff = abs(frb_icrs.ra.deg - expect_ra_deg)
dec_diff = abs(frb_icrs.dec.deg - expect_dec_deg)

print()
print("RA difference from expected:", ra_diff, "deg")
print("Dec difference from expected:", dec_diff, "deg")
print("Verified: RA and Dec match expected values within 1e-8 degree.")

RA (decimal degrees): 347.27041208333327 deg
Dec (decimal degrees): 48.70664105555556 deg
RA  (padded sexagesimal): 23:09:04.8989
Dec (padded sexagesimal): +48:42:23.9078

RA difference from expected: 3.325340003357269e-11 deg
Dec difference from expected: 4.4444448121794267e-11 deg
Verified: RA and Dec match expected values within 1e-8 degree.


RA and Dec both give the information about where an object in the sky, but they measure different units becuase RA tells us how far an object around the celestial equator. earth rotates , the sky appears to move across the sky once every 24 hrs. so RA is measured in hours from 0 to 24. this makes easy to know when an object will cross the meridian.

Dec tell us how far an object in north/south of celestial equator same as latitude of the earth. it measured in angle -90 deg to +90 deg.

## Q5. ICRS to Galactic and back [2 marks]

Transform `frb_icrs` to Galactic coordinates and print Galactic longitude $l$ and latitude $b$. Transform the result back to ICRS, calculate the round-trip separation in microarcseconds, and explain what a tiny non-zero residual represents.

In [11]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
frb_galactic = frb_icrs.galactic
print("Galactic longitude l:", frb_galactic.l.deg, "deg")
print("Galactic latitude b: ", frb_galactic.b.deg, "deg")

# Transform back to ICRS
frb_icrs_roundtrip = frb_galactic.icrs

# Round-trip separation in microarcseconds
separation = frb_icrs.separation(frb_icrs_roundtrip)
separation_uas = separation.to(u.microarcsecond)

print()
print("Round-trip separation:", separation_uas)

Galactic longitude l: 106.06496077332801 deg
Galactic latitude b:  -10.784176285402493 deg

Round-trip separation: 0.000310733 uarcsec


When I convert coodinates from icrs to galactic coordinates and again convert back to icrs from galactic. The small diffrence is the computer calulation error because computer need finite precision for doing calculation, so round off error not the astronomical error.

## Q6. ICRS to Ecliptic and back [2 marks]

Transform `frb_icrs` to barycentric true ecliptic coordinates at equinox J2000. Print ecliptic longitude $\lambda$ and latitude $\beta$ in degrees, verify their valid ranges, transform the result back to ICRS, and calculate the round-trip separation in microarcseconds. Explain what the ecliptic plane and the value of $\beta$ mean physically.

In [12]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
frb_ecliptic = frb_icrs.transform_to(BarycentricTrueEcliptic(equinox="J2000"))
print("Ecliptic longitude (lambda):", frb_ecliptic.lon.deg, "deg")
print("Ecliptic latitude (beta):  ", frb_ecliptic.lat.deg, "deg")
# Verify valid ranges: lambda in [0, 360) deg, beta in [-90, 90] deg
lon_deg = frb_ecliptic.lon.deg
lat_deg = frb_ecliptic.lat.deg

assert 0.0 <= lon_deg < 360.0, "Ecliptic longitude out of valid range [0, 360) deg!"
assert -90.0 <= lat_deg <= 90.0, "Ecliptic latitude out of valid range [-90, 90] deg!"
print()
print("Verified: ecliptic longitude is in [0, 360) deg and latitude is in [-90, 90] deg.")

# Transform back to ICRS
frb_icrs_roundtrip = frb_ecliptic.transform_to("icrs")

# Round-trip separation in microarcseconds
separation = frb_icrs.separation(frb_icrs_roundtrip)
separation_uas = separation.to(u.microarcsecond)

print()
print("Round-trip separation:", separation_uas)

Ecliptic longitude (lambda): 14.411187650340095 deg
Ecliptic latitude (beta):   48.34695843744783 deg

Verified: ecliptic longitude is in [0, 360) deg and latitude is in [-90, 90] deg.

Round-trip separation: 0 uarcsec


The ecliptic plane is the plane in which Earth moves around the Sun. If we look at the sky from Earth, this plane appears as the path along which the Sun seems to move during the year.

The ecliptic latitude (beta) tells us how far an object is above or below this plane:

Beta = 0° → object is exactly on the ecliptic plane.

Beta > 0° → object is north/above the ecliptic.

Beta < 0° → object is south/below the ecliptic.

Larger Beta → object is farther away from the ecliptic.

So, if Beta ≈ 48.3°, the FRB is very far from the ecliptic plane. It is nowhere near the usual path of the Sun and planets in the sky.

## Q7. Angular separation [2 marks]

Construct `earlier_position` from `23h09m04.9s`, `+48d42m25.4s`. Calculate the great-circle separation manually from

$$\cos(\Delta\theta)=\sin\delta_1\sin\delta_2+\cos\delta_1\cos\delta_2\cos(\alpha_2-\alpha_1).$$

Use radians inside NumPy, clip the computed cosine to $[-1,1]$, and report $\Delta\theta$ in arcseconds and milliarcseconds. Compare with `frb_icrs.separation(earlier_position)` and require agreement within $10^{-6}$ arcsecond.

In [13]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
earlier_position = SkyCoord(
    ra=Angle(EARLIER_POSITION_RA), dec=Angle(EARLIER_POSITION_DEC), frame="icrs"
)
alpha1 = np.longdouble(frb_icrs.ra.to_value(u.rad))
delta1 = np.longdouble(frb_icrs.dec.to_value(u.rad))
alpha2 = np.longdouble(earlier_position.ra.to_value(u.rad))
delta2 = np.longdouble(earlier_position.dec.to_value(u.rad))

cos_dtheta = (
    np.sin(delta1) * np.sin(delta2)
    + np.cos(delta1) * np.cos(delta2) * np.cos(alpha2 - alpha1)
)
cos_dtheta_clipped = np.clip(cos_dtheta, -1.0, 1.0)

delta_theta_rad = np.arccos(cos_dtheta_clipped)
delta_theta_arcsec = float(delta_theta_rad) * u.rad.to(u.arcsec) * u.arcsec
delta_theta_mas = delta_theta_arcsec.to(u.mas)

print("Manual separation (arcsec):", delta_theta_arcsec)
print("Manual separation (mas):   ", delta_theta_mas)

# Comparing
astropy_separation = frb_icrs.separation(earlier_position)
print()
print("Astropy separation (arcsec):", astropy_separation.to(u.arcsec))
agreement_diff = abs(delta_theta_arcsec - astropy_separation.to(u.arcsec))
print()
print("Difference:", agreement_diff)
assert agreement_diff < 1e-6 * u.arcsec, "Manual and Astropy separations disagree by more than 1e-6 arcsec!"
print("Verified: manual and Astropy separations agree within 1e-6 arcsecond.")

Manual separation (arcsec): 1.4922397272430614 arcsec
Manual separation (mas):    1492.2397272430612 mas

Astropy separation (arcsec): 1.49224 arcsec

Difference: 1.03961e-09 arcsec
Verified: manual and Astropy separations agree within 1e-6 arcsecond.


# Part C. GTC geometry, LST, transit, and altitude calculations (12 marks)

## Q8. Define the GTC observing site [1 mark]

Construct `gtc_location` with `EarthLocation.from_geodetic`, attaching units to the frozen longitude, latitude, and height. Print the recovered geodetic values. State what the negative longitude sign means.

In [14]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
gtc_location = EarthLocation.from_geodetic(
    lon=GTC_LONGITUDE_DEG * u.deg,
    lat=GTC_LATITUDE_DEG * u.deg,
    height=GTC_HEIGHT_M * u.m,
)

recovered_geodetic = gtc_location.to_geodetic()

print("GTC longitude:", recovered_geodetic.lon)
print("GTC latitude: ", recovered_geodetic.lat)
print("GTC height:   ", recovered_geodetic.height)

GTC longitude: -17d53m20.4s
GTC latitude:  28d45m28.8s
GTC height:    2395.9999999997463 m


Longitude tells us how far a place is east or west of the Greenwich meridian.

Positive longitude → the place is east of meridian.

Negative longitude → the place is west of meridian.

0 deg → the place is exactly on the Greenwich meridian.

So, the GTC has a longitude of -17.889°. The negative sign means that the telescope is about 17.9° west of meridian.

## Q9. Calculate GTC LST and hour angle at one instant [2 marks]

At `2026-08-31T00:00:00Z`, calculate apparent GTC Local Sidereal Time and target hour angle $H=\mathrm{LST}-\alpha$, wrapped at $\pm12$ hours. Print UTC, IIT Kanpur time, LST, and hour angle. From the sign of $H$, state whether the target is east of the meridian, transiting, or west of the meridian.

In [16]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
instant = Time("2026-08-31T00:00:00", scale="utc")

lst = instant.sidereal_time("apparent", longitude=gtc_location.lon)

# H = LST - RA,
hour_angle = (lst - frb_icrs.ra).wrap_at(12 * u.hourangle)

print("UTC:            ", format_utc(instant))
print("IIT Kanpur time:", format_iitk(instant))
print("LST (GTC):      ", lst)
print("Hour angle H:   ", hour_angle)

H_hours = hour_angle.hour
print()
if abs(H_hours) < 1e-6:
    print("H ~ 0: target is transiting (on the meridian).")
elif H_hours < 0:
    print("H < 0: target is EAST of the meridian (still rising toward transit).")
else:
    print("H > 0: target is WEST of the meridian (passed transit).")

UTC:             2026-08-31 00:00:00 UTC
IIT Kanpur time: 2026-08-31 05:30:00 IST
LST (GTC):       21h25m12.20821623s
Hour angle H:    -1h43m52.69068377s

H < 0: target is EAST of the meridian (still rising toward transit).


Hour Angle (H) tells us where an object is located compared to the local meridian (the imaginary line running from north to south through the sky).

The sign tells us whether the object is east or west of the meridian:

H < 0 → Object is east of the meridian. It has not reached the meridian yet and is moving toward it.

H = 0 → Object is exactly on the meridian. This is when it reaches its highest point in the sky (transit).

H > 0 → Object is west of the meridian. It has already passed the meridian and is moving toward setting.

## Q10. Find the nearest sampled meridian transit [2 marks]

Calculate apparent GTC LST and wrapped hour angle for all 145 samples. Locate the nearest sampled meridian transit by minimizing absolute hour angle. Print the sample index, UTC, IIT Kanpur time, LST, and hour angle. Explain why this is a sampled approximation rather than the exact transit instant.

In [18]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.

lst_all = times.sidereal_time("apparent", longitude=gtc_location.lon)
hour_angle_all = (lst_all - frb_icrs.ra).wrap_at(12 * u.hourangle)
transit_index = int(np.argmin(np.abs(hour_angle_all.hour)))

transit_time = times[transit_index]
transit_lst = lst_all[transit_index]
transit_H = hour_angle_all[transit_index]

print("Nearest sampled meridian transit:")
print("  Sample index:   ", transit_index)
print("  UTC:            ", format_utc(transit_time))
print("  IIT Kanpur time:", format_iitk(transit_time))
print("  LST (GTC):      ", transit_lst)
print("  Hour angle H:   ", transit_H)

Nearest sampled meridian transit:
  Sample index:    82
  UTC:             2026-08-31 01:40:00 UTC
  IIT Kanpur time: 2026-08-31 07:10:00 IST
  LST (GTC):       23h05m28.63507714s
  Hour angle H:    -0h03m36.26382286s


The result is a sampled approximation because the calculation checks (H) only at discrete 10-minute intervals, while the true transit occurs continuously at the exact instant when (H=0), which may lie between two sampled times.

## Q11. Predict transit altitude and zenith distance [2 marks]

For an upper meridian transit, use $h_{\mathrm{transit}}=90^\circ-|\phi-\delta|$, where $\phi$ is GTC latitude and $\delta$ is target declination. Calculate the predicted altitude and the lecture relation $z=90^\circ-h$ for zenith distance. Then transform the target to a vacuum `AltAz` frame for all grid times, find the largest sampled altitude and its zenith distance, and compare the analytic and Astropy values.

In [20]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
gtc_lat = gtc_location.lat
target_decl = frb_icrs.dec

h_transit = 90 * u.deg - np.abs(gtc_lat - target_decl)
z_transit = 90 * u.deg - h_transit

print("Analytic transit altitude h:      ", h_transit)
print("Analytic transit zenith distance z:", z_transit)

altaz_frame = AltAz(obstime=times, location=gtc_location, pressure=0 * u.hPa)
frb_altaz = frb_icrs.transform_to(altaz_frame)

max_index = int(np.argmax(frb_altaz.alt.deg))
max_altitude = frb_altaz.alt[max_index]
max_zenith_distance = 90 * u.deg - max_altitude

print()
print("Largest sampled altitude:      ", max_altitude, "at index", max_index)
print("Corresponding zenith distance: ", max_zenith_distance)

print()
print("Difference (altitude):        ", abs(h_transit - max_altitude))
print("Difference (zenith distance): ", abs(z_transit - max_zenith_distance))

Analytic transit altitude h:       70d03m04.8922s
Analytic transit zenith distance z: 19d56m55.1078s

Largest sampled altitude:       69d53m02.08616823s at index 82
Corresponding zenith distance:  20d06m57.91383177s

Difference (altitude):         0d10m02.80603177s
Difference (zenith distance):  0d10m02.80603177s


The analytic and Astropy values differ by about 10 arcminutes mainly because the analytic result uses a 10-minute time grid, so it may miss the exact transit and therefore slightly underestimate the maximum altitude. Astropy also uses a more precise coordinate transformation and Earth geometry than the simple analytic formula. Hence, the small difference is expected and it not indicate an error.

## Q12. Calculate altitude before, at, and after transit [3 marks]

At samples two hours before transit, nearest transit, and two hours after transit, calculate altitude from

$$\sin h=\sin\phi\sin\delta+\cos\phi\cos\delta\cos H.$$

Print UTC, IST, hour angle, calculated altitude, Astropy altitude, and their difference for all three samples. State the physical pattern you see as the source crosses the meridian.

In [21]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.

steps_per_2h = int(round((2 * u.hour / step).to_value(u.dimensionless_unscaled)))
sample_indices = [transit_index - steps_per_2h, transit_index, transit_index + steps_per_2h]
labels = ["2 hours before transit", "Nearest transit", "2 hours after transit"]

gtc_latitude_rad = gtc_location.lat.to_value(u.rad)
target_dec_rad = frb_icrs.dec.to_value(u.rad)

for label, idx in zip(labels, sample_indices):
    H_rad = hour_angle_all[idx].to_value(u.rad)
    sin_h = (
        np.sin(gtc_latitude_rad) * np.sin(target_dec_rad)
        + np.cos(gtc_latitude_rad) * np.cos(target_dec_rad) * np.cos(H_rad)
    )
    altitude_calc = (np.arcsin(np.clip(sin_h, -1.0, 1.0)) * u.rad).to(u.deg)
    altitude_astropy = frb_altaz.alt[idx]
    diff = abs(altitude_calc - altitude_astropy)

    print(label + ":")
    print("  UTC:            ", format_utc(times[idx]))
    print("  IIT Kanpur time:", format_iitk(times[idx]))
    print("  Hour angle H:   ", hour_angle_all[idx])
    print("  Calculated altitude:", altitude_calc)
    print("  Astropy altitude:   ", altitude_astropy)
    print("  Difference:         ", diff)
    print()

2 hours before transit:
  UTC:             2026-08-30 23:40:00 UTC
  IIT Kanpur time: 2026-08-31 05:10:00 IST
  Hour angle H:    -2h03m55.97605409s
  Calculated altitude: 59.031948665002126 deg
  Astropy altitude:    58d46m53.67263948s
  Difference:          0d15m01.34255453s

Nearest transit:
  UTC:             2026-08-31 01:40:00 UTC
  IIT Kanpur time: 2026-08-31 07:10:00 IST
  Hour angle H:    -0h03m36.26382286s
  Calculated altitude: 70.039347468167 deg
  Astropy altitude:    69d53m02.08616823s
  Difference:          0d09m19.56471717s

2 hours after transit:
  UTC:             2026-08-31 03:40:00 UTC
  IIT Kanpur time: 2026-08-31 09:10:00 IST
  Hour angle H:    1h56m43.44842115s
  Calculated altitude: 60.063043754805115 deg
  Astropy altitude:    60d09m45.39530037s
  Difference:          0d05m58.43778307s



As the FRB approaches the meridian, its altitude increases and reaches a maximum of about 70° at transit. After transit, its altitude decreases as it moves toward the west. The altitude changes very slowly near transit because the altitude curve is nearly flat at its maximum, while it changes faster farther from the meridian. The small 0.1–0.25° difference between the manual and Astropy results is expected because the manual formula uses simplified spherical geometry, whereas Astropy includes more detailed Earth and coordinate corrections.

## Q13. Calculate how long the source stays above 20 degrees [2 marks]

Solve the altitude equation for the limiting hour angle:

$$\cos H_0=\frac{\sin h_{\min}-\sin\phi\sin\delta}{\cos\phi\cos\delta}.$$

For $h_{\min}=20^\circ$, calculate $H_0$ in degrees and sidereal hours, then estimate the total interval $2H_0$ during which the source is above the altitude limit. Compare this with the number of grid samples above $20^\circ$.

In [23]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.

MIN_TARGET_ALTITUDE_DEG = 20.0

gtc_latitude_rad = gtc_location.lat.to_value(u.rad)
target_dec_rad = frb_icrs.dec.to_value(u.rad)
h_min_rad = (MIN_TARGET_ALTITUDE_DEG * u.deg).to_value(u.rad)

cos_H0 = (
    (np.sin(h_min_rad) - np.sin(gtc_latitude_rad) * np.sin(target_dec_rad))
    / (np.cos(gtc_latitude_rad) * np.cos(target_dec_rad))
)
cos_H0_clipped = np.clip(cos_H0, -1.0, 1.0)

H0_rad = np.arccos(cos_H0_clipped)
H0_deg = (H0_rad * u.rad).to(u.deg)
H0_sidereal_hours = (H0_rad * u.rad).to(u.hourangle)

print("Limiting hour angle H0 (degrees):       ", H0_deg)
print("Limiting hour angle H0 (sidereal hours):", H0_sidereal_hours)

total_interval_analytic = 2 * H0_sidereal_hours
print()
print("Estimated total interval above 20 deg (2*H0):", total_interval_analytic)

above_limit_mask = frb_altaz.alt.deg > MIN_TARGET_ALTITUDE_DEG
num_samples_above = int(np.sum(above_limit_mask))
sampled_interval = num_samples_above * STEP_MINUTES * u.min

print()
print("Number of grid samples above 20 deg:  ", num_samples_above)
print("Sampled interval above 20 deg (approx):", sampled_interval.to(u.hour))

Limiting hour angle H0 (degrees):        91.92747700350239 deg
Limiting hour angle H0 (sidereal hours): 6.12849846690016 hourangle

Estimated total interval above 20 deg (2*H0): 12.25699693380032 hourangle

Number of grid samples above 20 deg:   74
Sampled interval above 20 deg (approx): 12.333333333333334 h


# Part D. Airmass, Sun, Moon, and practical visibility (9 marks)

## Q14. Calculate a physically safe target airmass [2 marks]

Convert `frb_gtc_altaz.secz` to a float array named `target_airmass`. Replace entries with `NaN` whenever the target is at or below the horizon, the value is non-finite, or the value is below 1. Print the finite range, best airmass with UTC and IST, and the number of samples satisfying $1\leq X\leq2.5$. Explain why below-horizon values must not be used.

In [24]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.

altaz_frame = AltAz(obstime=times, location=gtc_location, pressure=0 * u.hPa)
frb_gtc_altaz = frb_icrs.transform_to(altaz_frame)

target_airmass = np.array(frb_gtc_altaz.secz, dtype=float)

below_horizon = frb_gtc_altaz.alt.deg <= 0
non_finite = ~np.isfinite(target_airmass)
below_one = target_airmass < 1

invalid_mask = below_horizon | non_finite | below_one
target_airmass[invalid_mask] = np.nan

finite_airmass = target_airmass[np.isfinite(target_airmass)]
print("Finite airmass range:", finite_airmass.min(), "to", finite_airmass.max())

best_index = int(np.nanargmin(target_airmass))
best_airmass = target_airmass[best_index]
best_time = times[best_index]

print()
print("Best (lowest) airmass:", best_airmass)
print("  UTC:            ", format_utc(best_time))
print("  IIT Kanpur time:", format_iitk(best_time))

within_limit_mask = (target_airmass >= 1) & (target_airmass <= MAX_AIRMASS)
num_within_limit = int(np.sum(within_limit_mask))
print()
print(f"Number of samples with 1 <= X <= {MAX_AIRMASS}:", num_within_limit)

Finite airmass range: 1.0649653072737555 to 55.33845064849153

Best (lowest) airmass: 1.0649653072737555
  UTC:             2026-08-31 01:40:00 UTC
  IIT Kanpur time: 2026-08-31 07:10:00 IST

Number of samples with 1 <= X <= 2.5: 68


Below-horizon airmass values must be ignored because they have no physical meaning. Although 1/(secz) mathematically becomes negative when ( z >90 deg), an object below the horizon is not visible from the telescope. Masking these values prevents calculations such as argmin() from incorrectly treating a negative airmass as the best observing condition.

### Moon-conditions

Use the following simplified **course classification**. These labels are provided for this assignment; they are not universal observatory definitions.

| Label | Rule |
|---|---|
| `Unknown` | At least one of Moon altitude, Moon–target separation, or illuminated fraction is non-finite. |
| `Gray` | All three inputs are finite and neither the `Bright` nor `Dark` rule below applies. |
| `Bright` | The illuminated fraction is at least 0.70 **and** Moon altitude is above $10^\circ$, **or** Moon–target separation is at most $45^\circ$. |
| `Dark` | Moon altitude is below $0^\circ$, **or** Moon–target separation is at least $90^\circ$ **and** illuminated fraction is at most 0.25. |

Apply `Bright` first and `Dark` second. Therefore, if both rules happen to pass, the final label is `Dark`.

## Q15. Calculate Sun and Moon observing conditions [2 marks]

Using Astropy's built-in ephemeris, calculate arrays of Sun altitude, Moon altitude, Moon airmass, Moon–target separation, and approximate illuminated fraction $f=(1-\cos\epsilon)/2$, where $\epsilon$ is geocentric Sun–Moon elongation. Keep Moon airmass only from 1 to 3 and above the horizon. Classify each sample as Dark, Bright, Gray, or Unknown using the printed course rules, then print the diagnostic values and Moon label at transit.

In [25]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested

with solar_system_ephemeris.set("builtin"):
    sun_gcrs = get_body("sun", times)
    moon_gcrs = get_body("moon", times)

    sun_altaz = sun_gcrs.transform_to(altaz_frame)
    moon_altaz = moon_gcrs.transform_to(altaz_frame)

sun_altitude = sun_altaz.alt
moon_altitude = moon_altaz.alt


moon_target_separation = moon_gcrs.separation(frb_icrs)

# illuminated fraction f = (1 - cos(eps)) / 2
sun_moon_elongation = sun_gcrs.separation(moon_gcrs)
illuminated_fraction = (1 - np.cos(sun_moon_elongation)) / 2

# Moon airmass
moon_airmass = np.array(moon_altaz.secz, dtype=float)
moon_below_horizon = moon_altitude.deg <= 0
moon_airmass_invalid = (
    moon_below_horizon | ~np.isfinite(moon_airmass)
    | (moon_airmass < 1) | (moon_airmass > 3)
)
moon_airmass[moon_airmass_invalid] = np.nan

labels = np.full(len(times), "Unknown", dtype=object)

moon_alt_deg = moon_altitude.deg
sep_deg = moon_target_separation.deg
frac = illuminated_fraction.value

finite_inputs = np.isfinite(moon_alt_deg) & np.isfinite(sep_deg) & np.isfinite(frac)
labels[finite_inputs] = "Gray"

bright_mask = finite_inputs & (
    ((frac >= 0.70) & (moon_alt_deg > 10)) | (sep_deg <= 45)
)
dark_mask = finite_inputs & (
    (moon_alt_deg < 0) | ((sep_deg >= 90) & (frac <= 0.25))
)

labels[bright_mask] = "Bright"
labels[dark_mask] = "Dark"   #

print("At transit sample (index", transit_index, "):")
print("  UTC:            ", format_utc(times[transit_index]))
print("  IIT Kanpur time:", format_iitk(times[transit_index]))
print("  Sun altitude:            ", sun_altitude[transit_index])
print("  Moon altitude:           ", moon_altitude[transit_index])
print("  Moon airmass:            ", moon_airmass[transit_index])
print("  Moon-target separation:  ", moon_target_separation[transit_index])
print("  Illuminated fraction:    ", frac[transit_index])
print("  Moon condition label:    ", labels[transit_index])

 '2026-08-30T12:20:00.000' '2026-08-30T12:30:00.000'
 '2026-08-30T12:40:00.000' '2026-08-30T12:50:00.000'
 '2026-08-30T13:00:00.000' '2026-08-30T13:10:00.000'
 '2026-08-30T13:20:00.000' '2026-08-30T13:30:00.000'
 '2026-08-30T13:40:00.000' '2026-08-30T13:50:00.000'
 '2026-08-30T14:00:00.000' '2026-08-30T14:10:00.000'
 '2026-08-30T14:20:00.000' '2026-08-30T14:30:00.000'
 '2026-08-30T14:40:00.000' '2026-08-30T14:50:00.000'
 '2026-08-30T15:00:00.000' '2026-08-30T15:10:00.000'
 '2026-08-30T15:20:00.000' '2026-08-30T15:30:00.000'
 '2026-08-30T15:40:00.000' '2026-08-30T15:50:00.000'
 '2026-08-30T16:00:00.000' '2026-08-30T16:10:00.000'
 '2026-08-30T16:20:00.000' '2026-08-30T16:30:00.000'
 '2026-08-30T16:40:00.000' '2026-08-30T16:50:00.000'
 '2026-08-30T17:00:00.000' '2026-08-30T17:10:00.000'
 '2026-08-30T17:20:00.000' '2026-08-30T17:30:00.000'
 '2026-08-30T17:40:00.000' '2026-08-30T17:50:00.000'
 '2026-08-30T18:00:00.000' '2026-08-30T18:10:00.000'
 '2026-08-30T18:20:00.000' '2026-08-30T18:30:0

At transit sample (index 82 ):
  UTC:             2026-08-31 01:40:00 UTC
  IIT Kanpur time: 2026-08-31 07:10:00 IST
  Sun altitude:             -51d57m21.32665026s
  Moon altitude:            59d05m45.35620029s
  Moon airmass:             1.1654625278023825
  Moon-target separation:   44d46m35.95410664s
  Illuminated fraction:     0.9081719277845963
  Moon condition label:     Bright


## Q16. Implement the visibility function [3 marks]

Write exactly

```python
visibility(altitude, airmass, sun_altitude, moon_separation)
```

It must validate four equal one-dimensional arrays and return one Boolean array. A sample is `True` only when altitude is at least $20^\circ$, airmass is finite and from 1 to 2.5, Sun altitude is at most $-18^\circ$, and Moon separation is at least $30^\circ$. Group every element-wise comparison with parentheses and join the results with `&`, not Python `and`. Print the independent condition counts, the combined count, and the overall yes/no answer.

In [26]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested
def visibility(altitude, airmass, sun_altitude, moon_separation):
    altitude = np.asarray(altitude, dtype=float)
    airmass = np.asarray(airmass, dtype=float)
    sun_altitude = np.asarray(sun_altitude, dtype=float)
    moon_separation = np.asarray(moon_separation, dtype=float)

    # Validate: all four arrays must share the same 1-D shape
    shapes = {altitude.shape, airmass.shape, sun_altitude.shape, moon_separation.shape}
    if len(shapes) != 1:
        raise ValueError("All four input arrays must have the same shape.")
    if altitude.ndim != 1:
        raise ValueError("All four input arrays must be one-dimensional.")

    altitude_ok = (altitude >= MIN_TARGET_ALTITUDE_DEG)
    airmass_ok = (np.isfinite(airmass)) & (airmass >= 1) & (airmass <= MAX_AIRMASS)
    sun_ok = (sun_altitude <= MAX_SUN_ALTITUDE_DEG)
    moon_ok = (moon_separation >= MIN_MOON_SEPARATION_DEG)

    combined = altitude_ok & airmass_ok & sun_ok & moon_ok

    print("Altitude condition count:       ", int(np.sum(altitude_ok)))
    print("Airmass condition count:        ", int(np.sum(airmass_ok)))
    print("Sun altitude condition count:   ", int(np.sum(sun_ok)))
    print("Moon separation condition count:", int(np.sum(moon_ok)))
    print("Combined (all four) count:      ", int(np.sum(combined)))
    print("Any sample visible overall:     ", bool(np.any(combined)))

    return combined


is_visible = visibility(
    frb_gtc_altaz.alt.deg,
    target_airmass,
    sun_altitude.deg,
    moon_target_separation.deg,
)

Altitude condition count:        74
Airmass condition count:         68
Sun altitude condition count:    51
Moon separation condition count: 145
Combined (all four) count:       51
Any sample visible overall:      True


## Q17. Does transit automatically mean visible? [2 marks]

Evaluate the target at (a) the nearest sampled transit from Q10 and (b) `2026-08-31T06:00:00Z`, when it is still passing through the western sky. For each instant, print UTC, IST, altitude, airmass, Sun altitude, Moon separation, the pass/fail result of every condition, and the final result from `preferred_visibility`. Explain why high altitude or meridian transit alone cannot guarantee observability.

In [27]:
def preferred_visibility(instant):

    altaz_frame = AltAz(obstime=instant, location=gtc_location, pressure=0 * u.hPa)
    target_altaz = frb_icrs.transform_to(altaz_frame)

    airmass = float(target_altaz.secz)
    if target_altaz.alt.deg <= 0 or not np.isfinite(airmass) or airmass < 1:
        airmass = np.nan

    with solar_system_ephemeris.set("builtin"):
        sun_gcrs = get_body("sun", instant)
        moon_gcrs = get_body("moon", instant)
    sun_altaz = sun_gcrs.transform_to(altaz_frame)
    moon_separation = moon_gcrs.separation(frb_icrs)

    # Individual pass/fail flags, computed the same way visibility() computes them internally
    altitude_ok = bool(target_altaz.alt.deg >= MIN_TARGET_ALTITUDE_DEG)
    airmass_ok = bool(np.isfinite(airmass) and (1 <= airmass <= MAX_AIRMASS))
    sun_ok = bool(sun_altaz.alt.deg <= MAX_SUN_ALTITUDE_DEG)
    moon_ok = bool(moon_separation.deg >= MIN_MOON_SEPARATION_DEG)

    # Final combined result: call visibility() itself, which returns ONE Boolean array
    combined = visibility(
        [target_altaz.alt.deg], [airmass], [sun_altaz.alt.deg], [moon_separation.deg]
    )

    return {
        "instant": instant, "altitude": target_altaz.alt, "airmass": airmass,
        "sun_altitude": sun_altaz.alt, "moon_separation": moon_separation,
        "altitude_ok": altitude_ok, "airmass_ok": airmass_ok,
        "sun_ok": sun_ok, "moon_ok": moon_ok, "visible": bool(combined[0]),
    }


evening_time = Time("2026-08-31T06:00:00", scale="utc")

for label, instant in [("(a) Nearest sampled transit", transit_time),
                        ("(b) 2026-08-31T06:00:00Z (western sky)", evening_time)]:
    result = preferred_visibility(instant)
    print(label)
    print("  UTC:             ", format_utc(result["instant"]))
    print("  IIT Kanpur time:  ", format_iitk(result["instant"]))
    print("  Altitude:         ", result["altitude"], " -> pass:", result["altitude_ok"])
    print("  Airmass:          ", result["airmass"], " -> pass:", result["airmass_ok"])
    print("  Sun altitude:     ", result["sun_altitude"], " -> pass:", result["sun_ok"])
    print("  Moon separation:  ", result["moon_separation"], " -> pass:", result["moon_ok"])
    print("  Final visibility: ", result["visible"])
    print()

Altitude condition count:        1
Airmass condition count:         1
Sun altitude condition count:    1
Moon separation condition count: 1
Combined (all four) count:       1
Any sample visible overall:      True
(a) Nearest sampled transit
  UTC:              2026-08-31 01:40:00 UTC
  IIT Kanpur time:   2026-08-31 07:10:00 IST
  Altitude:          69d53m02.08616823s  -> pass: True
  Airmass:           1.0649653072737555  -> pass: True
  Sun altitude:      -51d57m21.32665026s  -> pass: True
  Moon separation:   44d46m35.95410664s  -> pass: True
  Final visibility:  True

Altitude condition count:        1
Airmass condition count:         1
Sun altitude condition count:    0
Moon separation condition count: 1
Combined (all four) count:       0
Any sample visible overall:      False
(b) 2026-08-31T06:00:00Z (western sky)
  UTC:              2026-08-31 06:00:00 UTC
  IIT Kanpur time:   2026-08-31 11:30:00 IST
  Altitude:          37d59m11.35178503s  -> pass: True
  Airmass:           1.62

Transit only tells us that the target is at its maximum altitude and usually has minimum airmass. It does not  give the guarantee observability because other conditions, such as the Sun being below the astronomical-twilight limit, must also be satisfied. Thus, a target can have a good altitude and airmass but still be unobservable if the sky is too bright due to twilight.

# Part E. Observing and scheduling GTC (4 marks)

## Q18. Report the observing interval and make GTC airmass figure [4 marks]

Write `stitch_windows(times, visibility_state)` to return consecutive inclusive sample-centre intervals, including empty, isolated, separated, and final-sample cases. Apply it to `preferred_visibility`. Report each interval in IIT Kanpur time and UTC, its sample-centre duration, best target airmass, and midpoint Moon label.

Then make **one GTC airmass figure** and save it as `assignment_outputs/frb_20220912a_gtc_airmass.png`. Use a solid blue target curve, dashed gray Moon curve, an inverted 3-to-1 airmass axis, the 2.5 limit, scheduler-style twilight/night bands, green shading for calculated observable intervals, black markers for observable samples, IIT Kanpur time on the main x-axis, an aligned UTC axis above, a legend, date rollover, units, and an explanatory caption. State the final scientific conclusion and one limitation of the 10-minute grid.

In [28]:
def stitch_windows(times, visibility_state):
    """Return a list of (start_index, end_index) tuples marking consecutive
    inclusive runs of True in visibility_state, indexed into `times`.
    Handles: no True samples (empty list), single isolated True samples,
    multiple separated runs, and a run that reaches the final sample."""
    visibility_state = np.atleast_1d(np.asarray(visibility_state, dtype=bool))
    n = visibility_state.shape[0]
    if len(times) != n:
        raise ValueError(
            f"times has length {len(times)} but visibility_state has length {n}; "
            "they must match. Check that visibility_state is the full per-sample "
            "Boolean array (e.g. from visibility(...)), not a single scalar result."
        )
    intervals = []
    i = 0
    while i < n:
        if not visibility_state[i]:
            i += 1
            continue
        start = i
        j = i
        while j + 1 < n and visibility_state[j + 1]:
            j += 1
        intervals.append((start, j))
        i = j + 1
    return intervals


visibility_state = visibility(
    frb_gtc_altaz.alt.deg, target_airmass, sun_altitude.deg, moon_target_separation.deg
)
print("visibility_state shape:", np.asarray(visibility_state).shape)  # should print (145,)

windows = stitch_windows(times, visibility_state)

print("Observing windows:")
for k, (start_idx, end_idx) in enumerate(windows, start=1):
    window_start_time = times[start_idx]
    window_end_time = times[end_idx]
    duration = (window_end_time - window_start_time).to(u.hour)

    finite_airmass_in_window = target_airmass[start_idx:end_idx + 1]
    finite_airmass_in_window = finite_airmass_in_window[np.isfinite(finite_airmass_in_window)]
    best_airmass_in_window = np.min(finite_airmass_in_window)

    midpoint_index = start_idx + (end_idx - start_idx) // 2
    midpoint_label = labels[midpoint_index]

    print(f"Window {k}:")
    print("  Start (IST):", format_iitk(window_start_time), " | Start (UTC):", format_utc(window_start_time))
    print("  End   (IST):", format_iitk(window_end_time), " | End   (UTC):", format_utc(window_end_time))
    print("  Duration (sample-centre):", duration)
    print("  Best airmass in window:  ", best_airmass_in_window)
    print("  Midpoint Moon label:     ", midpoint_label)

Altitude condition count:        74
Airmass condition count:         68
Sun altitude condition count:    51
Moon separation condition count: 145
Combined (all four) count:       51
Any sample visible overall:      True
visibility_state shape: (145,)
Observing windows:
Window 1:
  Start (IST): 2026-08-31 02:30:00 IST  | Start (UTC): 2026-08-30 21:00:00 UTC
  End   (IST): 2026-08-31 10:50:00 IST  | End   (UTC): 2026-08-31 05:20:00 UTC
  Duration (sample-centre): 8.333333333333332 h
  Best airmass in window:   1.0649653072737555
  Midpoint Moon label:      Bright


In [29]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

times_ist_dt = [t.to_datetime(timezone=IITK_TIMEZONE) for t in times]

fig, ax = plt.subplots(figsize=(12, 7))

# Sun-altitude-based day/twilight/night bands
sun_alt_deg = sun_altitude.deg
night_mask = sun_alt_deg <= MAX_SUN_ALTITUDE_DEG
astro_twilight_mask = (sun_alt_deg > -18) & (sun_alt_deg <= -12)
nautical_twilight_mask = (sun_alt_deg > -12) & (sun_alt_deg <= -6)
civil_twilight_mask = (sun_alt_deg > -6) & (sun_alt_deg <= 0)
day_mask = sun_alt_deg > 0

def shade_regions(ax, mask, color, alpha, label):
    n = len(mask); i = 0; first = True
    while i < n:
        if not mask[i]:
            i += 1; continue
        start = i; j = i
        while j + 1 < n and mask[j + 1]:
            j += 1
        ax.axvspan(times_ist_dt[start], times_ist_dt[j], color=color, alpha=alpha,
                   label=label if first else None, zorder=0)
        first = False; i = j + 1

shade_regions(ax, day_mask, "gold", 0.15, "Day")
shade_regions(ax, civil_twilight_mask, "orange", 0.15, "Civil twilight")
shade_regions(ax, nautical_twilight_mask, "darkorange", 0.15, "Nautical twilight")
shade_regions(ax, astro_twilight_mask, "slategray", 0.20, "Astronomical twilight")
shade_regions(ax, night_mask, "navy", 0.15, "Astronomical night")
shade_regions(ax, visibility_state, "green", 0.25, "Observable window")   # <-- was preferred_visibility

ax.plot(times_ist_dt, target_airmass, color="blue", linestyle="-", linewidth=2,
        label=f"{FRB_NAME} airmass", zorder=3)
ax.plot(times_ist_dt, moon_airmass, color="gray", linestyle="--", linewidth=1.5,
        label="Moon airmass", zorder=2)

observable_indices = np.where(visibility_state)[0]   # <-- was preferred_visibility
ax.scatter([times_ist_dt[i] for i in observable_indices],
           [target_airmass[i] for i in observable_indices],
           color="black", s=18, zorder=4, label="Observable samples")
ax.axhline(MAX_AIRMASS, color="red", linestyle=":", linewidth=1.5,
           label=f"Airmass limit ({MAX_AIRMASS})")

ax.set_ylim(3, 1)  # inverted 3-to-1 axis
ax.set_ylabel("Airmass (sec z)")
ax.set_xlabel("IIT Kanpur time (IST)")
ax.set_title(f"{FRB_NAME} airmass and visibility over GTC — night of {times_ist_dt[0].strftime('%Y-%m-%d')}")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M\n%d %b", tz=IITK_TIMEZONE))
ax.xaxis.set_major_locator(mdates.HourLocator(interval=2, tz=IITK_TIMEZONE))
ax.grid(True, alpha=0.3)

# Aligned UTC axis on top
ax_utc = ax.twiny()
ax_utc.set_xlim(ax.get_xlim())
utc_ticks_ist = ax.get_xticks()
ax_utc.set_xticks(utc_ticks_ist)
utc_tick_labels = [mdates.num2date(t, tz=IITK_TIMEZONE).astimezone(timezone.utc).strftime("%H:%M\n%d %b")
                    for t in utc_ticks_ist]
ax_utc.set_xticklabels(utc_tick_labels)
ax_utc.set_xlabel("UTC")

ax.legend(loc="upper right", fontsize=8, framealpha=0.9)

caption = (
    f"Airmass vs. time for {FRB_NAME} (blue, solid) and the Moon (gray, dashed) as seen from GTC "
    f"({GTC_LATITUDE_DEG:.3f}N, {abs(GTC_LONGITUDE_DEG):.3f}W, {GTC_HEIGHT_M:.0f} m) on the night of "
    f"{times_ist_dt[0].strftime('%Y-%m-%d')} IST. The y-axis is inverted (airmass 3 at bottom, 1 at top) "
    "so that better observing conditions appear higher on the plot. Shaded bands mark Sun-altitude-based "
    "day/twilight/night periods; green shading and black markers highlight samples satisfying all four "
    f"visibility criteria (altitude >= {MIN_TARGET_ALTITUDE_DEG:.0f} deg, airmass <= {MAX_AIRMASS}, "
    f"Sun altitude <= {MAX_SUN_ALTITUDE_DEG:.0f} deg, Moon separation >= {MIN_MOON_SEPARATION_DEG:.0f} deg). "
    "The red dotted line marks the airmass = 2.5 observability limit."
)
fig.text(0.5, -0.05, caption, ha="center", va="top", fontsize=8, wrap=True)

fig.tight_layout()
fig.savefig("assignment_outputs/frb_20220912a_gtc_airmass.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# Part F. All-sky observability (2 assessed marks + 2 optional bonus marks)

The previous parts established the physical ingredients needed for an observing decision. The next questions reuse those same ideas at increasing scale: first one source on one date, then a weekly all-sky GTC visibility animation over one year.

For Q19–Q20, use the geometric/airmass GTC visibility rule developed in the notebook: the source or sky position must be above the horizon, satisfy the GTC airmass limit, and be inside astronomical night. Do not include the Moon in these questions.


## Q19. Find the GTC visibility window of the repeating FRB on one date [2 marks]

Treat FRB 20220912A as a repeating source at its fixed ICRS/J2000 position. For `2026-09-01`, calculate its visibility from GTC across the full UTC day using the same HEALPix-style visibility logic: astronomical night, target altitude above the horizon, and target airmass no larger than 2.5. Report the start and end of each contiguous visibility interval in UTC and IIT Kanpur time, together with the total visible duration in hours.

This is a **single source**, not an all-sky calculation.


In [30]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.

# Full UTC calendar day: 2026-09-01T00:00:00 -> 2026-09-02T00:00:00
day_start = Time("2026-09-01T00:00:00", scale="utc")
day_stop = Time("2026-09-02T00:00:00", scale="utc")
step = STEP_MINUTES * u.min
number_of_steps = int(np.round(((day_stop - day_start) / step).to_value(u.dimensionless_unscaled)))
day_times = day_start + np.arange(number_of_steps + 1) * step

altaz_frame = AltAz(obstime=day_times, location=gtc_location, pressure=0 * u.hPa)
frb_altaz = frb_icrs.transform_to(altaz_frame)

target_airmass = np.array(frb_altaz.secz, dtype=float)
below_horizon = frb_altaz.alt.deg <= 0
target_airmass[below_horizon | ~np.isfinite(target_airmass) | (target_airmass < 1)] = np.nan

sun_altaz = get_sun(day_times).transform_to(altaz_frame)
sun_altitude = sun_altaz.alt

night_ok = sun_altitude.deg <= MAX_SUN_ALTITUDE_DEG
altitude_ok = frb_altaz.alt.deg > 0
airmass_ok = np.isfinite(target_airmass) & (target_airmass <= MAX_AIRMASS)

is_visible = night_ok & altitude_ok & airmass_ok

windows = stitch_windows(day_times, is_visible)

total_hours = 0.0
print(f"GTC visibility windows for {FRB_NAME} on 2026-09-01 (UTC day):")
for k, (s, e) in enumerate(windows, start=1):
    dur = (day_times[e] - day_times[s]).to(u.hour)
    total_hours += dur.value
    print(f"Window {k}:")
    print("  Start UTC:", format_utc(day_times[s]), " | Start IST:", format_iitk(day_times[s]))
    print("  End   UTC:", format_utc(day_times[e]), " | End   IST:", format_iitk(day_times[e]))
    print("  Duration:", dur)
    print()

print("Total visible duration (sum of windows):", total_hours, "hours")

GTC visibility windows for FRB 20220912A on 2026-09-01 (UTC day):
Window 1:
  Start UTC: 2026-09-01 00:00:00 UTC  | Start IST: 2026-09-01 05:30:00 IST
  End   UTC: 2026-09-01 05:20:00 UTC  | End   IST: 2026-09-01 10:50:00 IST
  Duration: 5.333333333333333 h

Window 2:
  Start UTC: 2026-09-01 21:00:00 UTC  | Start IST: 2026-09-02 02:30:00 IST
  End   UTC: 2026-09-02 00:00:00 UTC  | End   IST: 2026-09-02 05:30:00 IST
  Duration: 3.0 h

Total visible duration (sum of windows): 8.333333333333332 hours


## Q20. Generate a one-year GTC visibility animation [2 bonus marks]

Use the same **all-sky GTC visibility** model over the next year, starting on **2026-08-30** and sampled every 7 days. With a weekly cadence, the final sample within the 365-day window is **2027-08-29**. For each date, compute the HEALPix visibility map using the same GTC astronomical-night, altitude, and airmass conditions developed earlier.

Save each weekly map as a PNG frame and combine the frames into a GIF. The animation should show how the observable sky from GTC changes through the year. Use a moderate HEALPix resolution so that the calculation is practical.

Briefly describe what changes across the animation and why the observable region changes with date.

In [31]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
import imageio.v2 as imageio

MAX_AIRMASS = 2.5
MAX_SUN_ALTITUDE_DEG = -18.0
NSIDE = 32

FRAMES_DIR = OUTPUT_DIR / "frames"
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

observer = Observer(location=gtc_location, name="GTC")

npix = hp.nside2npix(NSIDE)
theta, phi = hp.pix2ang(NSIDE, np.arange(npix))
pixel_ra_deg = np.degrees(phi)
pixel_dec_deg = 90.0 - np.degrees(theta)
sky_grid = SkyCoord(ra=pixel_ra_deg * u.deg, dec=pixel_dec_deg * u.deg, frame="icrs")

start_date = Time("2026-08-30T00:00:00", scale="utc")
week_offsets_days = np.arange(0, 365, 7)
weekly_dates = start_date + week_offsets_days * u.day

frame_paths = []
for i, date in enumerate(weekly_dates):
    midnight_time = observer.midnight(date, which="next")
    altaz_frame = AltAz(obstime=midnight_time, location=gtc_location, pressure=0 * u.hPa)

    sky_altaz = sky_grid.transform_to(altaz_frame)
    sun_altaz = get_sun(midnight_time).transform_to(altaz_frame)

    airmass = np.array(sky_altaz.secz, dtype=float)
    alt_ok = sky_altaz.alt.deg > 0
    airmass_ok = np.isfinite(airmass) & (airmass >= 1) & (airmass <= MAX_AIRMASS)
    night_ok = bool(sun_altaz.alt.deg <= MAX_SUN_ALTITUDE_DEG)

    visible_map = (alt_ok & airmass_ok & night_ok).astype(float)

    fig = plt.figure(figsize=(8, 5))
    hp.mollview(
        visible_map, fig=fig.number,
        title=f"GTC all-sky visibility — {midnight_time.iso[:10]} (local midnight)",
        cmap="Greens", min=0, max=1, cbar=False,
        unit="Observable (1) / Not observable (0)", coord="C",
    )
    hp.graticule()
    frame_path = FRAMES_DIR / f"frame_{i:03d}.png"
    fig.savefig(frame_path, dpi=100, bbox_inches="tight")
    plt.close(fig)
    frame_paths.append(frame_path)

# Combine into a GIF
gif_path = OUTPUT_DIR / "gtc_visibility_year.gif"
frames = [imageio.imread(p) for p in frame_paths]
imageio.mimsave(gif_path, frames, duration=0.25, loop=0)

/usr/local/lib/python3.13/dist-packages/healpy/visufunc.py:225: UserWarning: Ignoring specified arguments in this call because figure with num: 1 already exists
  f = pylab.figure(fig, figsize=(8.5, 5.4))


## Submission checklist

- Restart the kernel and run every cell from top to bottom.
- Keep units attached during physical calculations.
- Use UTC internally; show IIT Kanpur time whenever reporting a civil clock time.
- Label Local Sidereal Time as LST, never IST.
- Keep the exact `visibility(altitude, airmass, sun_altitude, moon_separation)` interface.
- Confirm `assignment_outputs/frb_20220912a_gtc_airmass.png` exists and is non-empty.
- Confirm the Q20 one-year GTC GIF and its generated weekly frame PNGs are present.

## Compact glossary

- **Astropy quantity:** a numerical value stored together with a physical unit.
- **UTC:** the standard time scale used to identify an instant worldwide.
- **IST:** Indian Standard Time, the IIT Kanpur civil clock, UTC+05:30.
- **JD/MJD:** continuous astronomical day counts; MJD is JD minus 2,400,000.5.
- **ICRS:** a modern standard equatorial frame expressed with RA and Dec.
- **Galactic coordinates:** longitude $l$ and latitude $b$ referred to the Milky Way.
- **Ecliptic coordinates:** longitude $\lambda$ and latitude $\beta$ referred to Earth's orbital plane; longitude begins at the vernal equinox.
- **Angular separation:** the shortest great-circle angle between two directions on the celestial sphere.
- **Mollweide projection:** an equal-area elliptical map of the full celestial sphere; Matplotlib expects longitude and latitude in radians.
- **LST:** Local Sidereal Time, an astronomical angle determined by time and longitude.
- **Hour angle:** $H=\mathrm{LST}-\alpha$; zero at meridian transit.
- **Transit:** passage across the local meridian; usually the highest geometric altitude.
- **AltAz:** a horizon frame determined by observer location and time.
- **Zenith distance:** angular distance from the zenith, $z=90^\circ-h$.
- **Airmass:** approximate atmospheric path length relative to the zenith.
- **Astronomical darkness:** Sun altitude at or below $-18^\circ$ in this assignment.
- **Observable interval:** consecutive samples satisfying every printed condition.

- **HEALPix:** a hierarchical equal-area pixelization of the celestial sphere; each pixel can store an astronomical quantity such as visibility time.
- **Visibility time:** the amount of time a source satisfies the observing constraints during a specified interval.
- **Repeating FRB:** an FRB that can produce bursts on multiple occasions, so a future observing schedule can be planned for its fixed sky position.
